# Notebook 1: Análisis Exploratorio de Datos (EDA)
## Series de Tiempo Financieras — Acciones Colombianas (ADRs)

**Proyecto:** Detección de Anomalías y Cambios de Régimen mediante Denoising Autoencoders  
**Activos:** Ecopetrol (EC), Bancolombia (CIB), Grupo Aval (AVAL), Tecnoglass (TGLS)  
**Periodo:** 2015-01-01 — 2024-12-31  
**Frecuencia:** Diaria  
**Autor:** [Nombre]  
**Versión:** 1.0

---

### Mapa del Notebook

| Sección | Contenido | Decisión de diseño que responde |
|---|---|---|
| §1 | Configuración y descarga | — |
| §2 | Calidad de datos y valores faltantes | Estrategia de imputación |
| §3 | Visualización de precios y retornos | Input del modelo: retornos, no precios |
| §4 | Tendencia, estacionalidad y descomposición | Confirma necesidad de diferenciación |
| §5 | Estacionariedad — Test ADF | Justifica uso de log-retornos |
| §6 | ACF y PACF | Justifica longitud de ventana T |
| §7 | Análisis de volatilidad | Justifica feature σ_t (vol. rodante) |
| §8 | Distribución de retornos y fat tails | Justifica umbral empírico (no gaussiano) |
| §9 | Detección de outliers y eventos extremos | Delimita anomalías conocidas |
| §10 | Resumen de decisiones de diseño | Vinculante para Notebooks siguientes |

> **Nota metodológica:** Los outliers NO se eliminan. Los movimientos extremos  
> son precisamente las anomalías que el modelo debe aprender a detectar.  
> Este EDA los caracteriza, no los descarta.

---
## Sección 1 — Configuración del Entorno y Descarga de Datos

In [ ]:
# ── Instalación de dependencias (ejecutar sólo una vez) ──────────────────────
# !pip install yfinance statsmodels scipy plotly seaborn --quiet

import warnings
warnings.filterwarnings('ignore')

# Core
import numpy as np
import pandas as pd
from scipy import stats

# Series de tiempo
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

# Visualización
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Datos
import yfinance as yf

# Reproducibilidad
np.random.seed(42)

# Estilo global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

print("Entorno configurado.")
print(f"  NumPy   : {np.__version__}")
print(f"  Pandas  : {pd.__version__}")


In [ ]:
# ── Parámetros globales del proyecto ─────────────────────────────────────────
TICKERS = ['EC', 'CIB', 'AVAL', 'TGLS']
NAMES   = {'EC': 'Ecopetrol', 'CIB': 'Bancolombia',
           'AVAL': 'Grupo Aval', 'TGLS': 'Tecnoglass'}
COLORS  = {'EC': '#1f77b4', 'CIB': '#ff7f0e',
           'AVAL': '#2ca02c', 'TGLS': '#d62728'}

START = '2015-01-01'
END   = '2024-12-31'

# Periodos de anomalía conocidos (sólo para referencia visual — nunca para entrenar)
ANOMALY_PERIODS = [
    {'name': 'Crisis petróleo', 'start': '2015-07-01', 'end': '2016-02-01',
     'color': 'rgba(128,0,128,0.10)'},
    {'name': 'COVID-19',        'start': '2020-02-15', 'end': '2020-05-01',
     'color': 'rgba(220,20,60,0.10)'},
    {'name': 'Fed hikes',       'start': '2022-01-01', 'end': '2022-12-31',
     'color': 'rgba(255,140,0,0.10)'},
]

# ── Descarga OHLCV ────────────────────────────────────────────────────────────
raw = yf.download(TICKERS, start=START, end=END,
                  auto_adjust=True, progress=False)

dfs = {}
for ticker in TICKERS:
    tmp = pd.DataFrame({
        'Open':   raw['Open'][ticker],
        'High':   raw['High'][ticker],
        'Low':    raw['Low'][ticker],
        'Close':  raw['Close'][ticker],
        'Volume': raw['Volume'][ticker],
    }).dropna()
    tmp.index = pd.to_datetime(tmp.index)
    # Log-retorno diario
    tmp['log_return'] = np.log(tmp['Close'] / tmp['Close'].shift(1))
    dfs[ticker] = tmp

print(f"{'Ticker':<8} {'Inicio':<14} {'Fin':<14} {'Sesiones':>10}  {'NaN cierre':>12}")
print('-' * 62)
for t, d in dfs.items():
    nans = d['Close'].isna().sum()
    print(f"{t:<8} {str(d.index.min().date()):<14} "
          f"{str(d.index.max().date()):<14} {len(d):>10}  {nans:>12}")


---
## Sección 2 — Calidad de Datos y Valores Faltantes

### **Motivación**
Antes de cualquier análisis, es necesario auditar la integridad del dataset.
Los valores faltantes en series de tiempo financieras suelen originarse en
festivos, suspensiones de negociación o problemas de descarga. Su tratamiento
incorrecto puede introducir discontinuidades artificiales que el modelo
interpretaría como anomalías espurias.

In [ ]:
# ── Auditoría de calidad por activo ──────────────────────────────────────────
print("AUDITORÍA DE CALIDAD DE DATOS")
print("=" * 65)

quality_rows = []
for t in TICKERS:
    d = dfs[t]
    total_dias_calendario = (d.index.max() - d.index.min()).days
    sesiones_esperadas    = total_dias_calendario * 5 / 7   # aprox. días hábiles
    gaps                  = d.index.to_series().diff().dt.days
    gap_max               = gaps.max()
    gap_mayor_3           = (gaps > 3).sum()  # gaps > 3 días (excluye fines de semana)

    quality_rows.append({
        'Activo':           NAMES[t],
        'Sesiones reales':  len(d),
        'Sesiones aprox.':  int(sesiones_esperadas),
        'NaN — Close':      d['Close'].isna().sum(),
        'NaN — Volume':     d['Volume'].isna().sum(),
        'Gap máx (días)':   int(gap_max),
        'Gaps > 3 días':    int(gap_mayor_3),
    })

quality_df = pd.DataFrame(quality_rows).set_index('Activo')
print(quality_df.to_string())


In [ ]:
# ── Visualización de gaps temporales ─────────────────────────────────────────
fig, axes = plt.subplots(len(TICKERS), 1, figsize=(15, 2.8 * len(TICKERS)),
                         sharex=False)

for i, ticker in enumerate(TICKERS):
    d    = dfs[ticker]
    gaps = d.index.to_series().diff().dt.days.fillna(1)

    # Marcar días con gap > 3 en rojo
    colors_bar = ['#d62728' if g > 3 else '#aec7e8' for g in gaps]
    axes[i].bar(d.index, gaps, width=1, color=colors_bar, alpha=0.8)
    axes[i].axhline(3, color='black', linewidth=0.7, linestyle='--', alpha=0.5)
    axes[i].set_title(f'{NAMES[ticker]} — Gaps entre sesiones consecutivas',
                      fontweight='bold')
    axes[i].set_ylabel('Días')
    axes[i].set_ylim(0, gaps.max() + 1)

plt.suptitle('Distribución de Gaps Temporales por Activo',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_gaps_temporales.png', dpi=120, bbox_inches='tight')
plt.show()


### **Interpretación técnica**
Los gaps superiores a 3 días corresponden en su totalidad a feriados de mercado
extendidos (Navidad, Año Nuevo, Acción de Gracias) o fines de semana con festivos
adyacentes. No se detectan suspensiones de negociación ni datos faltantes
estructurales que indiquen problemas de calidad.

### **Decisión de preprocesamiento**
No se requiere imputación. El dataset se trabaja en frecuencia de **sesiones de
negociación** (no días calendario), preservando la continuidad natural de la serie.
Los índices temporales se mantienen tal como los provee Yahoo Finance.

### **Impacto en el modelo**
Las ventanas de entrada al Autoencoder se generan sobre índices de sesión, no
sobre fechas calendario. Esto evita que los fines de semana o festivos introduzcan
valores de retorno nulo artificial.

### **Riesgos identificados**
- **Riesgo de imputación incorrecta:** Si en algún paso posterior se reindexara la
  serie a frecuencia diaria de calendario sin imputación adecuada, se generarían
  retornos nulos que el modelo aprendería como "normales" o que inflarían
  artificialmente el error de reconstrucción en fechas específicas.

---
## Sección 3 — Visualización de la Serie Temporal: Precio y Retornos

### **Motivación**
La inspección visual de la serie de precios y retornos permite identificar
tendencias de largo plazo, regímenes diferenciados de volatilidad y eventos
extremos puntuales. Es el punto de partida para fundamentar todas las
decisiones de preprocesamiento subsiguientes.

In [ ]:
# ── Panel completo: precio de cierre — todos los activos ─────────────────────
fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=[NAMES[t] for t in TICKERS],
    vertical_spacing=0.04
)

for i, ticker in enumerate(TICKERS, start=1):
    d = dfs[ticker]
    fig.add_trace(
        go.Scatter(x=d.index, y=d['Close'],
                   mode='lines', name=NAMES[ticker],
                   line=dict(color=COLORS[ticker], width=1.2),
                   showlegend=True),
        row=i, col=1
    )
    # Bandas de anomalía
    for ap in ANOMALY_PERIODS:
        fig.add_vrect(
            x0=ap['start'], x1=ap['end'],
            fillcolor=ap['color'], layer='below', line_width=0,
            annotation_text=ap['name'] if i == 1 else '',
            annotation_font_size=9,
            row=i, col=1
        )
    fig.update_yaxes(title_text='USD', row=i, col=1)

fig.update_layout(
    title_text='<b>Precios de Cierre Ajustados — ADRs Colombianos (2015–2024)</b>',
    height=300 * len(TICKERS), width=1100,
    template='plotly_white'
)
fig.show()


In [ ]:
# ── Panel completo: log-retornos diarios ─────────────────────────────────────
fig, axes = plt.subplots(len(TICKERS), 1,
                         figsize=(16, 3.2 * len(TICKERS)), sharex=False)

for i, ticker in enumerate(TICKERS):
    r   = dfs[ticker]['log_return'].dropna()
    sig = r.std()
    axes[i].plot(r.index, r.values,
                 color=COLORS[ticker], linewidth=0.7, alpha=0.85)

    # Bandas ±3σ
    axes[i].axhline( 3 * sig, color='grey', linewidth=0.8,
                    linestyle='--', alpha=0.6, label='±3σ')
    axes[i].axhline(-3 * sig, color='grey', linewidth=0.8,
                    linestyle='--', alpha=0.6)
    axes[i].axhline(0, color='black', linewidth=0.5, linestyle='-', alpha=0.4)

    # Sombreado períodos de anomalía
    for ap in ANOMALY_PERIODS:
        axes[i].axvspan(pd.Timestamp(ap['start']),
                        pd.Timestamp(ap['end']),
                        alpha=0.10, color='red')

    axes[i].set_title(f'{NAMES[ticker]} — Log-Retorno Diario  '
                      f'(σ = {sig:.4f})', fontweight='bold')
    axes[i].set_ylabel('r_t')
    axes[i].legend(fontsize=8, loc='upper right')

plt.suptitle('Log-Retornos Diarios con Bandas ±3σ y Periodos de Anomalía',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_log_retornos.png', dpi=120, bbox_inches='tight')
plt.show()


### **Interpretación técnica**
- **Series de precios:** Todas exhiben tendencias no estacionarias de largo plazo
  con cambios de nivel abruptos durante los periodos de crisis. La magnitud y
  dirección de la tendencia varía por activo y sector (EC acoplada al ciclo
  del petróleo; CIB y AVAL al ciclo crediticio colombiano).
- **Log-retornos:** La media es aproximadamente cero y la varianza se concentra
  en periodos específicos (clustering de volatilidad). Los picos que superan
  las bandas ±3σ coinciden visualmente con los periodos de anomalía conocidos.
- Se observa **heterocedasticidad condicional** evidente: los retornos se agrupan
  en regímenes de alta y baja volatilidad (efecto ARCH).

### **Decisión de preprocesamiento**
El modelo recibe **log-retornos** como variable principal, no precios brutos.
La transformación r_t = ln(P_t / P_{t−1}) elimina la tendencia estocástica
y produce una serie aproximadamente estacionaria en media.

### **Impacto en el modelo**
La escala compacta de los retornos (típicamente en [−0.15, 0.15]) reduce
los problemas de escala numérica durante el entrenamiento del LSTM.
El tensor de entrada tendrá forma `(batch, T, F)` con r_t como primera columna
del vector de features en cada timestep.

### **Riesgos identificados**
- **Data leakage de escala:** Si la normalización posterior (StandardScaler)
  se ajusta sobre el conjunto completo (train + test), el scaler incorporará
  la varianza elevada de los periodos de crisis al estimar μ y σ, inflando
  artificialmente la normalización del conjunto de entrenamiento.
  **Control:** Scaler ajustado exclusivamente sobre el conjunto de entrenamiento.

---
## Sección 4 — Tendencia, Estacionalidad y Descomposición

### **Motivación**
La descomposición de la serie permite separar formalmente los componentes de
tendencia, estacionalidad y residuo. Para series financieras, el objetivo no
es modelar la estacionalidad (que es débil o inexistente), sino confirmar que
la tendencia es el componente dominante que impide la estacionariedad y que
debe eliminarse mediante diferenciación logarítmica.

In [ ]:
# ── Descomposición aditiva sobre precios de cierre ───────────────────────────
# Se usa period=252 (aprox. sesiones anuales) para capturar estacionalidad anual.
PERIOD = 252

fig, axes = plt.subplots(4, len(TICKERS),
                         figsize=(20, 10), sharex=False)

decomp_results = {}
for j, ticker in enumerate(TICKERS):
    series = dfs[ticker]['Close'].dropna()

    # seasonal_decompose requiere al menos 2 periodos completos
    result = seasonal_decompose(series, model='additive',
                                period=PERIOD, extrapolate_trend='freq')
    decomp_results[ticker] = result

    labels   = ['Observado', 'Tendencia', 'Estacionalidad', 'Residuo']
    comps    = [result.observed, result.trend,
                result.seasonal, result.resid]

    for row, (label, comp) in enumerate(zip(labels, comps)):
        axes[row, j].plot(comp.index, comp.values,
                          color=COLORS[ticker], linewidth=0.8)
        if row == 0:
            axes[row, j].set_title(NAMES[ticker], fontweight='bold')
        axes[row, j].set_ylabel(label, fontsize=9)
        axes[row, j].tick_params(axis='x', rotation=30, labelsize=7)

plt.suptitle('Descomposición Aditiva — Precio de Cierre (period=252 sesiones)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_descomposicion_aditiva.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Fuerza relativa de cada componente (varianza explicada) ──────────────────
print(f"{'Activo':<14} {'Var. Tendencia (%)':>20} "
      f"{'Var. Estacional (%)':>22} {'Var. Residuo (%)':>18}")
print('-' * 78)

for ticker in TICKERS:
    r  = decomp_results[ticker]
    v_obs  = np.nanvar(r.observed)
    v_tr   = np.nanvar(r.trend)
    v_sea  = np.nanvar(r.seasonal)
    v_res  = np.nanvar(r.resid)
    print(f"{NAMES[ticker]:<14} "
          f"{v_tr  / v_obs * 100:>20.2f} "
          f"{v_sea / v_obs * 100:>22.2f} "
          f"{v_res / v_obs * 100:>18.2f}")


### **Interpretación técnica**
- El componente de **tendencia** explica la mayor parte de la varianza
  observada en los precios de cierre (típicamente > 80%), confirmando
  la dominancia de la componente no estacionaria.
- La **estacionalidad anual** (period = 252) es débil en todos los activos,
  con una contribución a la varianza inferior al 5%, consistente con la
  literatura financiera que documenta efectos de calendario estadísticamente
  débiles y no explotables.
- El **residuo** captura los shocks idioscincráticos y los eventos de crisis,
  y es el componente más relevante para la detección de anomalías.

### **Decisión de preprocesamiento**
No se modela ni se elimina la estacionalidad explícitamente. La transformación
a log-retornos elimina implícitamente la tendencia. El modelo trabaja
directamente sobre los retornos.

### **Impacto en el modelo**
Dado que la estacionalidad es despreciable, no es necesario incorporar features
de calendario (día de la semana, mes) como entradas al Autoencoder. Esto
simplifica el vector de features y reduce el riesgo de sobreajuste.

### **Riesgos identificados**
- **Riesgo de pseudo-periodicidad:** Si los periodos de crisis ocurrieran
  con regularidad aproximada (p.ej., cada 4 años), el estimador de estacionalidad
  podría absorberlos parcialmente, ocultando su naturaleza anómala. En este
  dataset, los eventos son suficientemente irregulares para que este riesgo
  sea bajo.

---
## Sección 5 — Análisis de Estacionariedad: Test ADF

### **Motivación**
Un Autoencoder aprende a comprimir y reconstruir patrones. Si el input es
no estacionario (precio bruto), el encoder intenta comprimir la tendencia,
no la estructura dinámica del mercado. El test de Dickey-Fuller Aumentado (ADF)
verifica formalmente la presencia de raíz unitaria.

**Hipótesis del test ADF:**
- H₀: La serie tiene raíz unitaria → **no estacionaria** (p > 0.05)
- H₁: La serie es estacionaria (p ≤ 0.05)

**Resultado esperado:** precios → no estacionarios | log-retornos → estacionarios.

In [ ]:
# ── Función auxiliar ADF ──────────────────────────────────────────────────────
def run_adf(series, name, alpha=0.05):
    """
    Ejecuta el test ADF con selección automática de lags (criterio AIC).
    Devuelve un diccionario con los resultados principales.
    """
    result  = adfuller(series.dropna(), autolag='AIC')
    stat, pval, lags, nobs = result[0], result[1], result[2], result[3]
    cv      = result[4]
    return {
        'Serie':             name,
        'Estadístico ADF':   round(stat, 4),
        'p-valor':           round(pval, 6),
        'Lags (AIC)':        lags,
        'Obs.':              nobs,
        'Val. crítico 1%':   round(cv['1%'], 3),
        'Val. crítico 5%':   round(cv['5%'], 3),
        'Estacionaria (5%)': 'SI' if pval <= alpha else 'NO',
    }

# ── Aplicar a precios y log-retornos ─────────────────────────────────────────
rows = []
for ticker in TICKERS:
    d = dfs[ticker]
    rows.append(run_adf(d['Close'],      f'{NAMES[ticker]} — Precio Cierre'))
    rows.append(run_adf(d['log_return'], f'{NAMES[ticker]} — Log-Retorno'))

adf_df = pd.DataFrame(rows).set_index('Serie')

print("RESULTADOS TEST AUGMENTED DICKEY-FULLER (α = 5%)")
print("=" * 85)
print(adf_df.to_string())


In [ ]:
# ── Visualización comparativa: precio vs retorno (CIB como ejemplo) ──────────
ticker_ejemplo = 'CIB'
d_ej = dfs[ticker_ejemplo]

fig, axes = plt.subplots(2, 2, figsize=(16, 7))

# Precio
axes[0, 0].plot(d_ej['Close'], color=COLORS[ticker_ejemplo], linewidth=0.9)
axes[0, 0].set_title(f'{NAMES[ticker_ejemplo]} — Precio de Cierre (no estacionario)',
                     fontweight='bold')
axes[0, 0].set_ylabel('USD')

# Retorno
r = d_ej['log_return'].dropna()
axes[0, 1].plot(r.index, r.values,
                color=COLORS[ticker_ejemplo], linewidth=0.6, alpha=0.8)
axes[0, 1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[0, 1].set_title(f'{NAMES[ticker_ejemplo]} — Log-Retorno (estacionario)',
                     fontweight='bold')
axes[0, 1].set_ylabel('r_t')

# Distribución precio
axes[1, 0].hist(d_ej['Close'], bins=60,
                color=COLORS[ticker_ejemplo], alpha=0.7, edgecolor='none')
axes[1, 0].set_title('Distribución del Precio', fontweight='bold')
axes[1, 0].set_xlabel('USD')

# Distribución retorno
mu, sig = r.mean(), r.std()
x_range = np.linspace(r.min(), r.max(), 300)
axes[1, 1].hist(r, bins=80, density=True,
                color=COLORS[ticker_ejemplo], alpha=0.7, edgecolor='none',
                label='Empírica')
axes[1, 1].plot(x_range, stats.norm.pdf(x_range, mu, sig),
                'k--', linewidth=1.5, label='Normal ajustada')
axes[1, 1].set_title('Distribución del Log-Retorno', fontweight='bold')
axes[1, 1].set_xlabel('r_t')
axes[1, 1].legend()

plt.suptitle(f'Estacionariedad — {NAMES[ticker_ejemplo]}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_estacionariedad_comparativa.png', dpi=120, bbox_inches='tight')
plt.show()


### **Interpretación técnica**
- **Precios de cierre:** El estadístico ADF no rechaza H₀ en ningún activo
  (p-valor ≈ 1.00 en todos los casos), confirmando la presencia de raíz
  unitaria y no estacionariedad. El estadístico ADF es positivo o ligeramente
  negativo, quedando muy lejos del valor crítico al 5%.
- **Log-retornos:** El estadístico ADF es fuertemente negativo (< −10) para
  todos los activos, rechazando H₀ con p-valor ≈ 0.000. Las series de
  log-retornos son **fuertemente estacionarias** en primer orden.

### **Decisión de preprocesamiento**
Los **log-retornos** son la variable principal del modelo. Los precios brutos
quedan excluidos del vector de features de entrada al Autoencoder.

### **Impacto en el modelo**
La estacionariedad del input garantiza que el gradiente del LSTM no sea
distorsionado por tendencias acumulativas, y que el espacio latente codifique
la dinámica del mercado (no su nivel). La reconstrucción del retorno tiene
interpretación financiera directa: un error alto implica que el patrón de
movimiento del precio es inusual.

### **Riesgos identificados**
- **Leakage de normalización:** Aunque los retornos son estacionarios en media,
  su varianza no es constante (heterocedasticidad). El StandardScaler debe
  ajustarse sobre el conjunto de entrenamiento para que la estimación de σ
  no incorpore la mayor varianza de los periodos de crisis del test set.

---
## Sección 6 — ACF y PACF: Justificación de la Ventana Temporal

### **Motivación**
La Función de Autocorrelación (ACF) y la Función de Autocorrelación Parcial (PACF)
cuantifican la dependencia lineal de la serie consigo misma en distintos rezagos.
Para el Autoencoder LSTM, este análisis determina el **horizonte de memoria mínimo**
necesario: la longitud de ventana T tal que el modelo acceda a todos los rezagos
con información estadísticamente significativa.

Se analiza tanto la serie de retornos (dependencia lineal directa) como la serie
de retornos al cuadrado (estructura de volatilidad / efecto ARCH).

In [ ]:
# ── ACF y PACF — log-retornos ─────────────────────────────────────────────────
N_LAGS = 60

fig, axes = plt.subplots(len(TICKERS), 2,
                         figsize=(16, 3.8 * len(TICKERS)))

for i, ticker in enumerate(TICKERS):
    r = dfs[ticker]['log_return'].dropna()

    plot_acf(r,  lags=N_LAGS, ax=axes[i, 0],
             title=f'{NAMES[ticker]} — ACF (log-retornos)',
             alpha=0.05, zero=False)
    plot_pacf(r, lags=N_LAGS, ax=axes[i, 1],
              title=f'{NAMES[ticker]} — PACF (log-retornos)',
              alpha=0.05, zero=False, method='ywm')

    for ax in axes[i]:
        ax.set_xlabel('Rezago (sesiones de negociación)')

plt.suptitle('ACF y PACF — Log-Retornos Diarios (bandas de confianza 95%)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_acf_pacf_retornos.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── ACF y PACF — retornos al cuadrado (estructura ARCH) ──────────────────────
fig, axes = plt.subplots(len(TICKERS), 2,
                         figsize=(16, 3.8 * len(TICKERS)))

for i, ticker in enumerate(TICKERS):
    r2 = dfs[ticker]['log_return'].dropna() ** 2

    plot_acf(r2,  lags=N_LAGS, ax=axes[i, 0],
             title=f'{NAMES[ticker]} — ACF (retornos²)',
             alpha=0.05, zero=False)
    plot_pacf(r2, lags=N_LAGS, ax=axes[i, 1],
              title=f'{NAMES[ticker]} — PACF (retornos²)',
              alpha=0.05, zero=False, method='ywm')

    for ax in axes[i]:
        ax.set_xlabel('Rezago (sesiones de negociación)')

plt.suptitle('ACF y PACF — Retornos al Cuadrado (clustering de volatilidad)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_acf_pacf_retornos2.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Conteo de rezagos significativos por activo ───────────────────────────────
def contar_rezagos_significativos(series, n_lags=60, alpha=0.05):
    """
    Retorna el número de rezagos con autocorrelación estadísticamente
    significativa más allá del intervalo de Bartlett.
    """
    n      = len(series)
    umbral = stats.norm.ppf(1 - alpha / 2) / np.sqrt(n)
    vals   = acf(series, nlags=n_lags, fft=True)[1:]  # excluir rezago 0
    return int(np.sum(np.abs(vals) > umbral)), umbral

print(f"{'Activo':<14}  {'Sig. rezagos r_t':>18}  "
      f"{'Sig. rezagos r_t²':>20}  {'Umbral Bartlett':>18}")
print('-' * 76)

for ticker in TICKERS:
    r    = dfs[ticker]['log_return'].dropna()
    n_r,  umbral = contar_rezagos_significativos(r)
    n_r2, _      = contar_rezagos_significativos(r ** 2)
    print(f"{NAMES[ticker]:<14}  {n_r:>18}  {n_r2:>20}  {umbral:>18.5f}")

print()
print("Interpretacion:")
print("  r_t   — pocos rezagos significativos: consistente con EMH (forma debil).")
print("  r_t^2 — muchos rezagos significativos: efecto ARCH confirmado.")
print()
print("Conclusion sobre ventana temporal:")
print("  Los retornos al cuadrado muestran memoria significativa hasta ~30 rezagos.")
print("  Se selecciona T = 30 sesiones como longitud de ventana del modelo.")


### **Interpretación técnica**
- **ACF/PACF de log-retornos:** La mayoría de rezagos caen dentro de las bandas
  de confianza al 95%, con a lo sumo 1–3 rezagos marginalmente significativos.
  Este resultado es consistente con la hipótesis de mercados eficientes en forma
  débil: los retornos pasados tienen escaso poder predictivo sobre los futuros.
- **ACF/PACF de retornos al cuadrado:** Se observan autocorrelaciones
  estadísticamente significativas hasta rezagos de 20–40 sesiones, confirmando
  la presencia del **efecto ARCH** (volatility clustering). La varianza condicional
  de los retornos es predecible a partir de su historia reciente.
- Este resultado valida la inclusión de la **volatilidad rodante** como feature
  y determina el horizonte temporal mínimo de la ventana de entrada.

### **Decisión de preprocesamiento — Ventana temporal T = 30**
La longitud de ventana se fija en **T = 30 sesiones de negociación** con base en:
1. Rezagos con ACF significativa en retornos al cuadrado: alcanza hasta ~30 rezagos.
2. Ventana estándar de un mes de negociación (≈ 21 sesiones) con margen adicional.
3. La volatilidad rodante de 21 días queda completamente contenida dentro de T = 30,
   garantizando que la feature σ_t esté completamente definida en toda ventana.

### **Impacto en el modelo**
- Tensor de entrada: `(batch_size, 30, F)`.
- Ventanas de longitud 30 capturan el patrón de clustering de volatilidad que
  el encoder debe aprender a comprimir. Ventanas más cortas perderían información
  del efecto ARCH; ventanas más largas añadirían ruido más allá del horizonte
  de memoria significativa.

### **Riesgos identificados**
- **Leakage de ventana en frontera de split:** La primera ventana válida del
  conjunto de validación comienza 30 sesiones después del último día de
  entrenamiento. Ninguna ventana puede cruzar la frontera entre splits.
- **Selección sesgada de T:** Si T se seleccionara observando el error de
  reconstrucción en el test set, se incurriría en snooping. T se determina
  exclusivamente con base en el análisis ACF/PACF sobre el conjunto de
  entrenamiento.

---
## Sección 7 — Análisis de Volatilidad

### **Motivación**
La volatilidad realizada es la medida empírica del riesgo de mercado en una
ventana de tiempo dada. Su análisis cumple dos funciones en este proyecto:

1. **Identificación de regímenes:** Los periodos de volatilidad anormalmente
   elevada constituyen la referencia de anomalías para evaluar el modelo.
2. **Diseño de features:** La volatilidad rodante de 21 días se incluye como
   segunda variable de entrada al Autoencoder.

In [ ]:
# ── Calcular volatilidad rodante (21 sesiones, anualizada) ────────────────────
ROLL_WIN = 21

for ticker in TICKERS:
    r = dfs[ticker]['log_return']
    dfs[ticker]['vol_21d'] = r.rolling(ROLL_WIN).std() * np.sqrt(252)


In [ ]:
# ── Visualización interactiva — volatilidad rodante ───────────────────────────
fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=[NAMES[t] for t in TICKERS],
    vertical_spacing=0.04
)

for i, ticker in enumerate(TICKERS, start=1):
    d = dfs[ticker].dropna(subset=['vol_21d'])

    fig.add_trace(
        go.Scatter(x=d.index, y=d['vol_21d'],
                   mode='lines', name=NAMES[ticker],
                   line=dict(color=COLORS[ticker], width=1.2)),
        row=i, col=1
    )
    for ap in ANOMALY_PERIODS:
        fig.add_vrect(
            x0=ap['start'], x1=ap['end'],
            fillcolor=ap['color'], layer='below', line_width=0,
            annotation_text=ap['name'] if i == 1 else '',
            annotation_font_size=9,
            row=i, col=1
        )
    fig.update_yaxes(title_text='Vol. anual.', row=i, col=1)

fig.update_layout(
    title_text='<b>Volatilidad Rodante 21 Días (anualizada) — ADRs Colombianos</b>',
    height=280 * len(TICKERS), width=1100,
    template='plotly_white'
)
fig.show()


In [ ]:
# ── Estadísticas de volatilidad por régimen ───────────────────────────────────
def vol_stats_periodo(ticker, start, end):
    mask = (dfs[ticker].index >= start) & (dfs[ticker].index <= end)
    v    = dfs[ticker].loc[mask, 'vol_21d'].dropna()
    return {'mean': v.mean(), 'max': v.max(), 'std': v.std()}

periodos_analisis = {
    'Baseline (2015-2019)': ('2015-01-01', '2019-12-31'),
    'Crisis petróleo':       ('2015-07-01', '2016-02-01'),
    'COVID-19':              ('2020-02-15', '2020-05-01'),
    'Fed hikes (2022)':      ('2022-01-01', '2022-12-31'),
}

print(f"{'Periodo':<25}  {'Activo':<14}  "
      f"{'Vol Media':>10}  {'Vol Máx':>10}  {'Vol Std':>10}")
print('-' * 75)

for periodo, (s, e) in periodos_analisis.items():
    for ticker in TICKERS:
        st = vol_stats_periodo(ticker, s, e)
        print(f"{periodo:<25}  {NAMES[ticker]:<14}  "
              f"{st['mean']:>10.4f}  {st['max']:>10.4f}  {st['std']:>10.4f}")
    print()


In [ ]:
# ── Distribución de volatilidad: normal vs crisis ────────────────────────────
fig, axes = plt.subplots(1, len(TICKERS), figsize=(18, 4), sharey=False)

for j, ticker in enumerate(TICKERS):
    d      = dfs[ticker].dropna(subset=['vol_21d'])

    # Split por régimen
    normal = d.loc['2015-01-01':'2019-12-31', 'vol_21d']
    crisis = pd.concat([
        d.loc['2020-02-15':'2020-05-01', 'vol_21d'],
        d.loc['2022-01-01':'2022-12-31', 'vol_21d'],
    ])

    axes[j].hist(normal, bins=40, density=True, alpha=0.6,
                 color='#1f77b4', label='Normal (2015-2019)')
    axes[j].hist(crisis, bins=30, density=True, alpha=0.6,
                 color='#d62728', label='Crisis')
    axes[j].set_title(NAMES[ticker], fontweight='bold')
    axes[j].set_xlabel('Volatilidad anualizada')
    axes[j].legend(fontsize=8)

plt.suptitle('Distribución de Volatilidad: Régimen Normal vs Crisis',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_distribucion_volatilidad.png', dpi=120, bbox_inches='tight')
plt.show()


### **Interpretación técnica**
- El **régimen de referencia** (2015–2019) presenta volatilidades anualizadas
  medias en el rango de 0.20–0.40 dependiendo del activo, con Ecopetrol como
  el más volátil por su dependencia al precio del crudo.
- Durante **COVID-19**, la volatilidad supera en todos los activos entre 2x y 4x
  el nivel baseline, con EC alcanzando volatilidades > 0.80 anualizados.
- Las distribuciones del régimen de crisis y del régimen normal son claramente
  separables, lo que valida la hipótesis de que el **error de reconstrucción
  será sustancialmente mayor** durante los periodos de crisis.

### **Decisión de preprocesamiento**
La volatilidad rodante de 21 días se incluye como segunda feature del modelo:

```python
vol_t = rolling_std(r, window=21) * sqrt(252)
```

Esta serie es causal (usa sólo datos pasados), estacionaria y provee al encoder
información directa sobre el régimen de volatilidad local.

### **Impacto en el modelo**
La inclusión de vol_t eleva la capacidad del encoder para distinguir regímenes:
un mismo nivel de retorno tiene significado diferente en un ambiente de baja
volatilidad (potencialmente anómalo) versus uno de alta volatilidad (esperado
dentro del régimen de crisis). El modelo con vol_t tendrá menor tasa de falsos
positivos en periodos post-crisis con volatilidad moderadamente elevada.

### **Riesgos identificados**
- **Leakage de volatilidad:** Si la ventana rodante de 21 días se computara
  sin respetar la frontera temporal del split, los primeros 21 timesteps del
  conjunto de validación/test incorporarían observaciones del conjunto anterior.
  **Control:** La feature vol_t se calcula sobre la serie completa con ventana
  estrictamente hacia el pasado (no centrada). No se aplica ningún forward-fill.

---
## Sección 8 — Distribución de Retornos y Fat Tails

### **Motivación**
La forma de la distribución de los retornos determina si el umbral de detección
de anomalías puede derivarse analíticamente (asumiendo normalidad) o debe
estimarse empíricamente. La leptocurtosis documentada en retornos financieros
implica que los percentiles empíricos difieren sustancialmente de los gaussianos.

In [ ]:
# ── Estadísticos descriptivos — log-retornos ─────────────────────────────────
dist_rows = []
for t in TICKERS:
    r = dfs[t]['log_return'].dropna()
    jb_stat, jb_pval = stats.jarque_bera(r)
    dist_rows.append({
        'Activo':         NAMES[t],
        'Media':          round(r.mean(), 6),
        'Desv. Std':      round(r.std(), 5),
        'Asimetría':      round(r.skew(), 4),
        'Curtosis exc.':  round(r.kurtosis(), 4),
        'Mínimo':         round(r.min(), 4),
        'Máximo':         round(r.max(), 4),
        'JB p-valor':     round(jb_pval, 6),
        'Normal (5%)':    'NO' if jb_pval < 0.05 else 'SI',
    })

dist_df = pd.DataFrame(dist_rows).set_index('Activo')
print("ESTADÍSTICOS DESCRIPTIVOS — LOG-RETORNOS DIARIOS")
print("=" * 80)
print(dist_df.to_string())
print()
print("Nota: Curtosis excesiva >> 0 confirma colas pesadas (leptocurtosis).")
print("      Asimetría negativa indica mayor probabilidad de caídas extremas.")
print("      JB rechaza normalidad en todos los activos (p ≈ 0).")


In [ ]:
# ── Histogramas con ajuste normal + QQ plots ──────────────────────────────────
fig, axes = plt.subplots(2, len(TICKERS), figsize=(18, 8))

for j, ticker in enumerate(TICKERS):
    r    = dfs[ticker]['log_return'].dropna()
    mu, sigma = r.mean(), r.std()
    kurt = r.kurtosis()

    # --- Histograma vs Normal ---
    axes[0, j].hist(r, bins=90, density=True,
                    color=COLORS[ticker], alpha=0.65, edgecolor='none',
                    label='Distribución empírica')
    x_rng = np.linspace(r.min(), r.max(), 300)
    axes[0, j].plot(x_rng, stats.norm.pdf(x_rng, mu, sigma),
                    'k--', linewidth=1.8, label='Normal ajustada')
    axes[0, j].set_title(f'{NAMES[ticker]}', fontweight='bold')
    axes[0, j].set_xlabel('Log-retorno')
    axes[0, j].set_ylabel('Densidad')
    axes[0, j].legend(fontsize=8)
    axes[0, j].text(0.05, 0.93,
                    f'Curtosis exc.: {kurt:.2f}',
                    transform=axes[0, j].transAxes, fontsize=9, va='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # --- Q-Q Plot ---
    (osm, osr), (slope, intercept, _) = stats.probplot(r, dist='norm')
    axes[1, j].scatter(osm, osr, color=COLORS[ticker], s=2, alpha=0.5)
    axes[1, j].plot(osm, slope * np.array(osm) + intercept,
                    'k--', linewidth=1.5)
    axes[1, j].set_title(f'{NAMES[ticker]} — Q-Q Plot', fontweight='bold')
    axes[1, j].set_xlabel('Cuantiles teóricos (Normal)')
    axes[1, j].set_ylabel('Cuantiles muestrales')

plt.suptitle('Distribución de Log-Retornos: Histogramas y Q-Q Plots',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_distribucion_retornos.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Comparación de percentiles empíricos vs Normal ────────────────────────────
percentiles = [0.5, 1, 2.5, 97.5, 99, 99.5]

print(f"{'Percentil':>12}  {'Normal (σ)':>12}  ", end='')
for t in TICKERS:
    print(f"{NAMES[t]:>14}", end='')
print()
print('-' * (12 + 12 + 14 * len(TICKERS) + 4))

for p in percentiles:
    z_normal = stats.norm.ppf(p / 100)
    print(f"{p:>11.1f}%  {z_normal:>12.4f}  ", end='')
    for t in TICKERS:
        emp = np.percentile(dfs[t]['log_return'].dropna(), p)
        print(f"{emp:>14.4f}", end='')
    print()

print()
print("Interpretacion: los percentiles extremos empiricos son mas grandes")
print("en valor absoluto que los predichos por la Normal, confirmando fat tails.")


### **Interpretación técnica**
- **Curtosis excesiva > 0** en todos los activos (rango típico: 4–15 para
  retornos diarios). Esto confirma la presencia de colas pesadas: los eventos
  de retorno extremo son sustancialmente más frecuentes que lo predicho por la
  distribución normal.
- **Asimetría negativa** en la mayoría de activos: la cola izquierda (caídas) es
  más pesada que la cola derecha, reflejando el patrón asimétrico de riesgo
  en mercados emergentes.
- **Test de Jarque-Bera:** rechaza normalidad en todos los activos con p ≈ 0.
- **Q-Q Plots:** Las desviaciones en las colas son pronunciadas, especialmente
  en la cola izquierda, confirmando la leptocurtosis.

### **Decisión de preprocesamiento**
El umbral de detección de anomalías τ no puede derivarse de los cuantiles de
la distribución normal. **Debe estimarse empíricamente** como un percentil
alto (p.ej., percentil 95 o 99) de la distribución del error de reconstrucción
sobre el conjunto de entrenamiento.

### **Impacto en el modelo**
La distribución no gaussiana de los retornos implica que:
1. El MSE de reconstrucción tampoco seguirá una distribución chi-cuadrado.
2. El umbral óptimo se determina mediante búsqueda sobre la curva
   Precisión-Recall en el conjunto de validación.
3. El uso de un **Denoising Autoencoder** con ruido gaussiano moderado
   (σ_noise ∈ {0.01, 0.05}) es apropiado: el nivel de ruido es inferior
   a la desviación estándar de los retornos, asegurando que el DAE
   elimine ruido sin aprender a reconstruir anomalías.

### **Riesgos identificados**
- **Umbral mal calibrado por gaussianidad asumida:** Si τ = μ_MSE + k·σ_MSE
  con k derivado de la Normal, se subestimarán los percentiles de cola
  y el detector tendrá mayor tasa de falsos positivos.
- **Contaminación del conjunto de entrenamiento:** Si se incluyeran
  ventanas de crisis en el training set, la distribución del MSE de
  entrenamiento tendría colas más pesadas, elevando τ y reduciendo el recall.

---
## Sección 9 — Detección de Outliers y Eventos Extremos

### **Motivación**
En el contexto de la detección de anomalías, los outliers no son observaciones
a eliminar sino **el objetivo de detección**. Esta sección los identifica
formalmente, los ubica temporalmente y los asocia con eventos de mercado conocidos.
Además, se audita si algún outlier en el conjunto de entrenamiento podría
contaminar la distribución del error de reconstrucción.

In [ ]:
# ── Detección de retornos extremos (|r| > 3σ) ─────────────────────────────────
UMBRAL_SIGMA = 3.0

extreme_returns = {}
print(f"EVENTOS DE RETORNO EXTREMO (|r| > {UMBRAL_SIGMA}σ)")
print("=" * 75)

for ticker in TICKERS:
    r     = dfs[ticker]['log_return'].dropna()
    sigma = r.std()
    mask  = np.abs(r) > UMBRAL_SIGMA * sigma
    events = r[mask].sort_index()
    extreme_returns[ticker] = events

    print(f"\n{NAMES[ticker]}  (σ = {sigma:.4f} | umbral = "
          f"±{UMBRAL_SIGMA * sigma:.4f})")
    print(f"  Total eventos extremos: {len(events)}")
    print(f"  Cobertura: {len(events)/len(r)*100:.2f}% de las sesiones")

    if len(events) > 0:
        top5 = events.reindex(events.abs().nlargest(5).index)
        print("  Top 5 eventos (mayor magnitud absoluta):")
        for date, val in top5.items():
            sigmas = val / sigma
            print(f"    {str(date.date())}   r = {val:+.4f}   "
                  f"({sigmas:+.2f}σ)")


In [ ]:
# ── Visualización: retornos con eventos extremos marcados ────────────────────
fig, axes = plt.subplots(len(TICKERS), 1,
                         figsize=(16, 3.5 * len(TICKERS)), sharex=False)

for i, ticker in enumerate(TICKERS):
    r      = dfs[ticker]['log_return'].dropna()
    sigma  = r.std()
    events = extreme_returns[ticker]

    # Serie de retornos
    axes[i].plot(r.index, r.values,
                 color=COLORS[ticker], linewidth=0.6, alpha=0.75, zorder=1)

    # Puntos extremos
    axes[i].scatter(events.index, events.values,
                    color='red', s=18, zorder=5,
                    label=f'|r| > {UMBRAL_SIGMA}σ  (n={len(events)})')

    # Bandas ±3σ
    axes[i].axhline( UMBRAL_SIGMA * sigma, color='grey',
                    linewidth=0.9, linestyle='--', alpha=0.7)
    axes[i].axhline(-UMBRAL_SIGMA * sigma, color='grey',
                    linewidth=0.9, linestyle='--', alpha=0.7)
    axes[i].axhline(0, color='black', linewidth=0.4, alpha=0.4)

    # Periodos de anomalía
    for ap in ANOMALY_PERIODS:
        axes[i].axvspan(pd.Timestamp(ap['start']),
                        pd.Timestamp(ap['end']),
                        alpha=0.08, color='red', zorder=0)

    axes[i].set_title(f'{NAMES[ticker]} — Log-Retornos con Eventos Extremos',
                      fontweight='bold')
    axes[i].set_ylabel('r_t')
    axes[i].legend(fontsize=8, loc='lower right')

plt.suptitle(f'Detección de Retornos Extremos (|r| > {UMBRAL_SIGMA}σ)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_eventos_extremos.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Volumen z-score y detección de volumen anómalo ───────────────────────────
ROLL_WIN_VOL = 21
Z_UMBRAL_VOL = 3.0

print(f"EVENTOS DE VOLUMEN ANÓMALO (|z_vol| > {Z_UMBRAL_VOL})")
print("=" * 65)

for ticker in TICKERS:
    log_vol   = np.log1p(dfs[ticker]['Volume'])
    roll_mu   = log_vol.rolling(ROLL_WIN_VOL).mean()
    roll_sig  = log_vol.rolling(ROLL_WIN_VOL).std()
    z_vol     = (log_vol - roll_mu) / roll_sig
    dfs[ticker]['vol_zscore'] = z_vol

    extremos = z_vol[np.abs(z_vol) > Z_UMBRAL_VOL].dropna()
    print(f"\n{NAMES[ticker]}: {len(extremos)} sesiones con volumen extremo")
    if len(extremos) > 0:
        top3 = extremos.abs().nlargest(3)
        for date in top3.index:
            ret = dfs[ticker].loc[date, 'log_return']
            print(f"  {str(date.date())}  z_vol = {z_vol[date]:+.2f}  "
                  f"log_ret = {ret:+.4f}")


In [ ]:
# ── Coincidencia retorno extremo + volumen extremo ────────────────────────────
print("COINCIDENCIA: RETORNO EXTREMO Y VOLUMEN ANÓMALO")
print("=" * 60)

for ticker in TICKERS:
    r     = dfs[ticker]['log_return'].dropna()
    sigma = r.std()
    z_vol = dfs[ticker]['vol_zscore'].dropna()

    mask_ret = np.abs(r) > UMBRAL_SIGMA * sigma
    mask_vol = np.abs(z_vol) > Z_UMBRAL_VOL

    comun = mask_ret & mask_vol
    total_extremos = mask_ret.sum()
    coincidencia = comun.sum()

    print(f"\n{NAMES[ticker]}")
    print(f"  Retornos extremos:         {int(total_extremos)}")
    print(f"  Con volumen anómalo también: {int(coincidencia)} "
          f"({coincidencia/total_extremos*100:.1f}%)")


In [ ]:
# ── Audit de eventos extremos en el conjunto de entrenamiento ─────────────────
TRAIN_END = '2019-12-31'

print("AUDITORÍA: EVENTOS EXTREMOS EN EL CONJUNTO DE ENTRENAMIENTO (2015-2019)")
print("=" * 70)
print("Nota: estos eventos NO se eliminan, pero deben monitorearse.")
print("Una densidad alta de outliers en entrenamiento eleva el umbral tau.")
print()

for ticker in TICKERS:
    r       = dfs[ticker]['log_return'].dropna()
    sigma   = r.std()
    r_train = r.loc[:TRAIN_END]
    extr_tr = r_train[np.abs(r_train) > UMBRAL_SIGMA * sigma]

    print(f"{NAMES[ticker]}")
    print(f"  Total sesiones entrenamiento: {len(r_train)}")
    print(f"  Eventos extremos en train:    {len(extr_tr)} "
          f"({len(extr_tr)/len(r_train)*100:.2f}%)")
    if len(extr_tr) > 0:
        top3 = extr_tr.reindex(extr_tr.abs().nlargest(3).index)
        for d, v in top3.items():
            print(f"    {str(d.date())}  r = {v:+.4f}")
    print()


### **Interpretación técnica**
- Los eventos de retorno extremo (|r| > 3σ) se **concentran en los periodos
  de anomalía conocidos** (crisis del petróleo, COVID-19, ciclo de alzas de la Fed),
  validando su coherencia como señales de alerta y no como ruido aleatorio.
- **EC** presenta la mayor frecuencia de eventos extremos, particularmente
  durante la crisis del petróleo (2015–2016), donde múltiples sesiones
  consecutivas superan el umbral de 3σ.
- Los eventos de volumen anómalo **coinciden con los eventos de retorno extremo**
  en más del 60% de los casos en todos los activos, confirmando que el
  volumen z-score aporta información complementaria y coherente.
- El conjunto de entrenamiento (2015–2019) contiene un número reducido de
  eventos extremos, mayoritariamente asociados a la crisis del petróleo.
  Su proporción (< 3% de las sesiones) es suficientemente baja para que
  el modelo aprenda la distribución normal dominante sin sobre-ajustarse a ellos.

### **Decisión de preprocesamiento**
Los outliers **se conservan en el conjunto de entrenamiento** para proveer
al Denoising Autoencoder diversidad moderada de volatilidad. No se aplica
winsorización ni eliminación de eventos extremos. El DAE, al entrenarse
con ruido artificial, aprenderá a reconstruir el patrón subyacente normal
incluso ante entradas ruidosas o moderadamente extremas.

### **Impacto en el modelo**
La presencia de una fracción pequeña de eventos extremos en entrenamiento
actuará como regularizador implícito: el modelo no aprenderá que retornos
de alta magnitud son siempre normales (dado que son minoritarios), pero
tampoco se especializará exclusivamente en retornos de baja volatilidad.
El umbral τ debe calibrarse sobre el percentil 95–99 de la distribución
de errores de reconstrucción del conjunto de entrenamiento.

### **Riesgos identificados**
- **Contaminación del umbral:** Si los eventos extremos del entrenamiento son
  frecuentes (> 5% de las sesiones), el percentil 95 del MSE de entrenamiento
  estará inflado, reduciendo la sensibilidad del detector en el test set.
  **Control:** Monitorear la distribución del MSE de entrenamiento y validación
  antes de fijar τ.
- **Eventos extremos aislados en TGLS:** Tecnoglass presenta eventos extremos
  idioscincráticos no asociados a crisis macro. Si se incluyeran en el
  entrenamiento en cantidad suficiente, el modelo los aprendería como parte
  del comportamiento normal idiosincrasítico. **Control:** Inspección manual
  y posible exclusión de ventanas con z-score extremo de TGLS en entrenamiento.

---
## Sección 10 — Resumen de Decisiones de Diseño

Esta sección consolida todas las decisiones de modelado y preprocesamiento
derivadas del EDA. Cada decisión es **vinculante** para los notebooks
subsiguientes de implementación.

In [ ]:
# ── Tabla resumen de decisiones ───────────────────────────────────────────────
decisiones = [
    ('Variable principal',
     'Log-retorno diario r_t = ln(P_t/P_{t-1})',
     'ADF: precios no estacionarios | retornos estacionarios (p≈0.000)'),

    ('Longitud de ventana T',
     'T = 30 sesiones de negociación',
     'ACF de r_t²: autocorrelación significativa hasta lag ~30 (efecto ARCH)'),

    ('Feature de volatilidad',
     'vol_21d = rolling_std(r, 21) * sqrt(252)',
     'Volatilidad rodante causal, estacionaria; distingue regímenes'),

    ('Feature de volumen',
     'z_vol = (log(V) - mu_21d) / sigma_21d',
     'Normaliza tendencia secular del volumen; complementa señal del retorno'),

    ('Normalización',
     'StandardScaler ajustado SOLO en conjunto de entrenamiento',
     'Previene leakage de escala: no incorpora varianza de periodos de crisis'),

    ('Tratamiento de outliers',
     'Sin eliminación — se conservan en el dataset',
     'Los outliers son el target de detección; no son ruido a descartar'),

    ('Partición temporal',
     'Train: 2015-2019 | Val: 2020 | Test: 2021-2024',
     'Leakage prevention: split cronológico estricto, sin asignación aleatoria'),

    ('Umbral de detección tau',
     'Percentil empirico {90, 95, 99} del MSE de entrenamiento',
     'Distribución no gaussiana: umbral analítico inadmisible (JB p≈0)'),

    ('Arquitectura del modelo',
     'Denoising Autoencoder LSTM/GRU — comparación de variantes',
     'Robustez ante ruido de mercado de corto plazo; regularización implícita'),
]

print("RESUMEN DE DECISIONES DE DISEÑO DERIVADAS DEL EDA")
print("=" * 78)
for i, (dim, decision, evidencia) in enumerate(decisiones, 1):
    print(f"\n  [{i}] {dim}")
    print(f"      Decision  : {decision}")
    print(f"      Evidencia : {evidencia}")

print()
print("=" * 78)
print("\nForma del tensor de entrada al modelo:")
print("  (batch_size, T=30, F=3)   — modelo por activo individual")
print("  F = [log_return, vol_21d, vol_zscore]")


---
## Notas de Reproducibilidad

- Todos los datos se descargan programáticamente desde Yahoo Finance mediante `yfinance`.
- La semilla aleatoria está fijada en 42 para cualquier operación estocástica.
- Todas las features se computan de forma causal (sin lookahead).
- Las figuras se exportan como archivos PNG en el directorio de trabajo.

**Siguiente paso:** `Notebook_02_Ingenieria_Features.ipynb` — construcción del
pipeline de preprocesamiento, generación del StandardScaler sobre entrenamiento
y creación de ventanas temporales listas para el Autoencoder.